# Detección temprana de displasia de cadera (DDC) con ResNet50 y GANs

Caso de estudio de ATLAS — AI Thesis Lab (FuzzyFrog.AI). Clasifica radiografías pélvicas
anteroposteriores de infantes de 3 a 6 meses como **DDC** o **Normal**.

**Pipeline:**
1. Preprocesamiento de imágenes
2. Exploración de separabilidad (K-Means + PCA)
3. Aumento de datos con una GAN (generador / discriminador)
4. Baseline: modelos clásicos (SVM, Árbol de Decisión) + CNN simple
5. Modelo final: ResNet50 con transfer learning y fine-tuning
6. Evaluación y comparación de modelos

> ⚠️ Este notebook es material educativo derivado de un proyecto de tesis. No debe usarse
> para diagnóstico clínico real sin validación adicional y supervisión de un especialista.


## 1. Configuración e imports

Dependencias principales del proyecto.

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score, auc
)

# Ruta base del dataset (ajustar a tu entorno)
DATA_DIR = './data/radiografias_pelvicas'
IMG_SIZE = (224, 224)
SEED = 42


## 2. Preprocesamiento

Cada radiografía se redimensiona a 224×224, se convierte a escala de grises, se aplica un
filtro Gaussiano 3×3 para reducir ruido, y se normalizan las intensidades a [0, 1]. Esto se
hace a través de `ImageDataGenerator`, para minimizar errores de carga y mantener el mismo
pipeline en entrenamiento y validación.

In [ ]:
def preprocess_input(x):
    """Filtro Gaussiano 3x3 aplicado antes de la normalización estándar del generador."""
    x = cv2.GaussianBlur(x, (3, 3), 0)
    return x

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rescale=1. / 255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=True,
    seed=SEED,
    subset='training'
)

valid_data = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary',
    color_mode='grayscale',
    seed=SEED,
    subset='validation'
)


## 3. Exploración de separabilidad (K-Means + PCA)

Antes de comprometerse con una arquitectura de clasificación, se valida que exista una señal
recuperable en las características de la imagen: K-Means sobre características simples
(promedio de intensidad por canal) y PCA para visualizar en 2D si los clusters resultantes
se parecen a las etiquetas reales.

In [ ]:
root_dir = DATA_DIR
carpetas = ['DDH', 'Normal']
identificadores = {'DDH': 0, 'Normal': 1}

caracteristicas, etiquetas_verdaderas = [], []

for carpeta in carpetas:
    ruta_carpeta = os.path.join(root_dir, carpeta)
    for filename in os.listdir(ruta_carpeta):
        if filename.endswith(('.jpg', '.png')):
            img = cv2.imread(os.path.join(ruta_carpeta, filename))
            if img is not None:
                feature_vector = np.mean(img, axis=(0, 1)).reshape(1, -1)
                caracteristicas.append(feature_vector)
                etiquetas_verdaderas.append(identificadores[carpeta])

X = np.vstack(caracteristicas)
y = np.array(etiquetas_verdaderas)

kmeans = KMeans(n_clusters=len(carpetas), random_state=0)
labels = kmeans.fit_predict(X)

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X)

plt.figure(figsize=(10, 6))
for i, carpeta in enumerate(carpetas):
    plt.scatter(principal_components[labels == i, 0], principal_components[labels == i, 1],
                label=f'Cluster {i+1}')
    plt.scatter(principal_components[y == identificadores[carpeta], 0],
                principal_components[y == identificadores[carpeta], 1],
                label=f'Etiqueta {carpeta}', alpha=0.5)
plt.title('Clustering de imágenes vs. etiquetas verdaderas')
plt.legend()
plt.show()

print('Matriz de confusión (clustering vs. etiqueta real):')
print(confusion_matrix(y, labels))
print('\nInforme de clasificación:')
print(classification_report(y, labels))


## 4. Aumento de datos con GAN

El dataset real está desbalanceado (120 DDC / 234 Normal, sobre 354 sujetos). En vez de
oversampling clásico (rotar, voltear, recortar), se entrena una GAN para generar radiografías
sintéticas nuevas y balancear ambas clases a 350 sujetos cada una (230 DDC sintéticas + 120
reales, 116 Normal sintéticas + 234 reales).

**Nota:** entrenar la GAN es costoso; una vez generadas las imágenes sintéticas y guardadas en
disco, esta sección no necesita volver a ejecutarse en corridas posteriores del pipeline.

In [ ]:
from keras.models import Sequential, Model
from keras.layers import Dense, Reshape, Conv2DTranspose, BatchNormalization, LeakyReLU, Conv2D, Flatten, Input
from keras.optimizers import Adam

IMG_ROWS, IMG_COLS, CHANNELS = 224, 224, 3
IMG_SHAPE = (IMG_ROWS, IMG_COLS, CHANNELS)
Z_DIM = 100  # dimensión del vector de ruido


def build_generator(z_dim):
    model = Sequential()
    model.add(Dense(128 * 56 * 56, input_dim=z_dim))
    model.add(Reshape((56, 56, 128)))
    model.add(Conv2DTranspose(64, kernel_size=3, strides=2, padding='same'))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.01))
    model.add(Conv2DTranspose(3, kernel_size=3, strides=2, padding='same', activation='tanh'))
    return model


def build_discriminator(img_shape):
    model = Sequential()
    model.add(Conv2D(64, kernel_size=3, strides=2, input_shape=img_shape, padding='same'))
    model.add(LeakyReLU(alpha=0.01))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    return model


discriminator = build_discriminator(IMG_SHAPE)
discriminator.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['accuracy'])

generator = build_generator(Z_DIM)
discriminator.trainable = False

z = Input(shape=(Z_DIM,))
img = generator(z)
valid = discriminator(img)

gan = Model(z, valid)
gan.compile(loss='binary_crossentropy', optimizer=Adam())


In [ ]:
def train_gan(generator, discriminator, gan, data_generator, epochs, batch_size, save_interval):
    real = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):
        # --- Entrena el discriminador ---
        imgs_real = next(data_generator)[0]
        noise = np.random.normal(0, 1, (batch_size, Z_DIM))
        imgs_fake = generator.predict(noise)

        d_loss_real = discriminator.train_on_batch(imgs_real, real)
        d_loss_fake = discriminator.train_on_batch(imgs_fake, fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # --- Entrena el generador ---
        noise = np.random.normal(0, 1, (batch_size, Z_DIM))
        g_loss = gan.train_on_batch(noise, real)

        print(f"{epoch} [D loss: {d_loss[0]:.4f}, acc.: {100 * d_loss[1]:.2f}%] [G loss: {g_loss:.4f}]")

        if epoch % save_interval == 0:
            save_synthetic_samples(generator, epoch)


def save_synthetic_samples(generator, epoch, out_dir='images'):
    os.makedirs(out_dir, exist_ok=True)
    r, c = 5, 5
    noise = np.random.normal(0, 1, (r * c, Z_DIM))
    gen_imgs = generator.predict(noise)
    gen_imgs = 0.5 * gen_imgs + 0.5

    fig, axs = plt.subplots(r, c)
    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i, j].imshow(gen_imgs[cnt, :, :, :])
            axs[i, j].axis('off')
            cnt += 1
    fig.savefig(f"{out_dir}/xray_{epoch}.png")
    plt.close()


# Descomentar para (re)entrenar la GAN. Costoso: correr en GPU.
# train_gan(generator, discriminator, gan, train_data, epochs=2000, batch_size=32, save_interval=200)


## 5. Baseline: modelos clásicos y CNN simple

Antes de ResNet50, se establece un punto de comparación honesto: SVM y Árbol de Decisión
sobre características planas de la imagen, y una CNN simple entrenada desde cero. La ganancia
de ResNet50 debe medirse contra esto, no asumirse.

In [ ]:
# Aplana los generadores de imágenes en arrays para SVM y Árbol de Decisión
X_train, y_train, X_valid, y_valid = [], [], [], []

for data_batch, labels_batch in train_data:
    X_train.extend(np.array(data_batch).reshape(len(data_batch), -1))
    y_train.extend(labels_batch)
    if len(X_train) >= len(train_data.classes):
        break

for data_batch, labels_batch in valid_data:
    X_valid.extend(np.array(data_batch).reshape(len(data_batch), -1))
    y_valid.extend(labels_batch)
    if len(X_valid) >= len(valid_data.classes):
        break

svm_classifier = SVC(kernel='rbf', probability=True)
svm_classifier.fit(X_train, y_train)

dt_classifier = DecisionTreeClassifier(random_state=SEED)
dt_classifier.fit(X_train, y_train)

for name, clf in [('SVM', svm_classifier), ('Árbol de Decisión', dt_classifier)]:
    pred_valid = clf.predict(X_valid)
    print(f"--- {name} ---")
    print("Exactitud:", accuracy_score(y_valid, pred_valid))
    print("Precisión:", precision_score(y_valid, pred_valid))
    print("Sensibilidad:", recall_score(y_valid, pred_valid))
    print("F1:", f1_score(y_valid, pred_valid), "\n")


In [ ]:
# CNN simple como baseline profundo
cnn_baseline = keras.Sequential([
    keras.layers.Conv2D(40, (3, 3), activation='relu', input_shape=(224, 224, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(40, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(40, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation='sigmoid'),
])

cnn_baseline.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_baseline = cnn_baseline.fit(
    train_data,
    validation_data=valid_data,
    epochs=15
)

test_loss, test_accuracy = cnn_baseline.evaluate(valid_data)
print(f'Exactitud del CNN baseline en validación: {test_accuracy:.4f}')


## 6. Modelo final: ResNet50 con transfer learning

Se carga ResNet50 preentrenado en ImageNet, se congela el modelo base y se entrena solo la
cabeza nueva. Después se liberan las últimas 20 capas para fine-tuning con un learning rate
mucho más bajo, evitando destruir los pesos preentrenados con gradientes grandes sobre un
dataset pequeño.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    validation_split=0.2
)

train_data_rn = train_datagen.flow_from_directory(
    DATA_DIR, target_size=(224, 224), batch_size=32,
    class_mode='binary', shuffle=True, seed=SEED, subset='training'
)
valid_data_rn = valid_datagen.flow_from_directory(
    DATA_DIR, target_size=(224, 224), batch_size=32,
    class_mode='binary', seed=SEED, subset='validation'
)

# Carga ResNet50 preentrenado en ImageNet, sin la cabeza de clasificación
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_data_rn, validation_data=valid_data_rn, epochs=10, batch_size=32)


In [ ]:
# Fine-tuning: libera las últimas 20 capas del modelo base con un LR mucho más bajo
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=0.00001), loss='binary_crossentropy', metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
fine_tune_epochs = 5
total_epochs = history.epoch[-1] + fine_tune_epochs

history_fine = model.fit(
    train_data_rn,
    validation_data=valid_data_rn,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1],
    callbacks=[early_stopping]
)


## 7. Evaluación y comparación de modelos

Métricas finales reportadas en el artículo (Tabla 2), con el CNN baseline como referencia:

| Métrica | CNN baseline | ResNet50 |
| --- | --- | --- |
| Exactitud | 74.29% | 97.43% |
| Sensibilidad | 73.74% | 98.82% |
| Precisión | 75.43% | 96.00% |
| AUC-ROC | 0.74 | 0.97 |


In [ ]:
y_pred = model.predict(valid_data_rn)
y_true = valid_data_rn.classes

y_pred_labels = (y_pred > 0.5).astype(int)

print("Exactitud:", accuracy_score(y_true, y_pred_labels))
print("Precisión:", precision_score(y_true, y_pred_labels))
print("Sensibilidad:", recall_score(y_true, y_pred_labels))

cm = confusion_matrix(y_true, y_pred_labels)
class_names = ['DDC', 'Normal']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Clase predicha')
plt.ylabel('Clase verdadera')
plt.title('Matriz de confusión — ResNet50')
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ResNet50 (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar (AUC = 0.50)')
plt.xlabel('Tasa de falsos positivos')
plt.ylabel('Tasa de verdaderos positivos (sensibilidad)')
plt.title('Curva ROC — ResNet50 vs. baseline')
plt.legend()
plt.show()


## 8. Conclusiones

- La combinación de GAN (balanceo de datos) + ResNet50 (transfer learning) mejora
  sustancialmente sobre el baseline: de 74.3% a 97.4% de exactitud.
- Ningún modelo clásico (SVM, Árbol de Decisión) ni el CNN simple superó el 75% de exactitud,
  lo que confirma que la ganancia viene de la arquitectura y del balanceo combinados.
- **Limitación honesta:** el dataset real detrás de estas métricas son 354 sujetos de solo dos
  hospitales. El resultado es evidencia de que el pipeline funciona, no de que el modelo
  generaliza a otras poblaciones, equipos o protocolos de radiografía. Antes de cualquier uso
  clínico se necesitaría validación externa en más centros.
- **Trabajo futuro:** probar otras arquitecturas (Inception, VGG, DenseNet), y extender el
  modelo de clasificación binaria a clasificación de severidad de la DDC.
